# Xhakinha Clips AI — notebook operacional Kaggle (continuidade Turso + B2)

## Regras de execução
Execute as etapas na ordem, uma célula por vez. A numeração operacional considera somente células de código; a quantidade total de células pode variar quando houver necessidade técnica.

Este é o único notebook operacional: `kaggle/kaggle_worker.ipynb`. O artefato R3 permanece auxiliar de auditoria; não é um segundo Worker e não deve ser executado em paralelo.

- Turso é o estado canônico; B2 privado é o armazenamento de objetos e snapshots.
- A exportação/dry-run usa uma cópia SQLite isolada; não transforma SQLite local em banco canônico.
- A tabela `sources` pode estar vazia nesta fase. A consulta e o schema precisam ser válidos; fontes serão cadastradas/autorizadas pelo painel.
- A etapa 15B pode criar o primeiro snapshot no B2 se ainda não houver um manifesto válido. Não apaga nem sobrescreve snapshots existentes.
- As etapas 15C e 15D verificam lock, executores, tarefas, snapshot e gates.
- A inicialização real permanece desativada por padrão. Não altere `INICIAR_KAGGLE_WORKER = False` durante auditorias.

| Etapa (código) | Função |
|---|---|
| 1 | Sincronização e dependências |
| 2 | Secrets e configuração Turso/B2 |
| 3 | Healthcheck e validação direta Turso |
| 4 | Imports e dependências |
| 5 | Estado local/Turso — somente leitura |
| 6 | Coordenador e baseline |
| 7 | B2, quota e snapshots — somente leitura |
| 8 | Lock e executores |
| 9 | Exportação Turso para cópia isolada |
| 10 | Fontes e scheduler — dry-run |
| 11 | Integridade e idempotência |
| 12 | Relatório e pré-requisitos |
| 13 | Status atual |
| 14 | Executores — observação |
| 15 | Pré-validação final |
| 16 | 15B — bootstrap/validação do snapshot B2 |
| 17 | 15C — conferência remota final |
| 18 | 15D — gate final somente de leitura |
| 19 | Inicialização manual — desativada por padrão |

Não reprocessar a fonte aprovada, não duplicar os seis clipes existentes e não publicar conteúdo durante a auditoria.

## 1) Inicialização e sincronização do repositório — prepara o ambiente (clone/pull + pip; idempotente)

In [ ]:
import os, sys, subprocess, base64, hashlib, json, time, socket, builtins
from pathlib import Path

REPO   = "https://github.com/gav74742-star/Xhakinha.git"
BRANCH = "main"
BASE   = Path(os.environ.get("XH_NB_BASE", "/kaggle/working/app"))
CLONE  = BASE / "xhakinha-clips-ai"

# Cada execução da auditoria começa com resultados novos. Preservamos apenas
# o hash das dependências para evitar reinstalação desnecessária neste kernel.
_XH_ANTERIOR = globals().get("XH", {})
_XH_DEPS_HASH = (
    _XH_ANTERIOR.get("deps_hash")
    if isinstance(_XH_ANTERIOR, dict) else None
)
XH = {
    "res": {},
    "audit_version": "2026-11-25-r3",
    "audit_started_at": time.time(),
}
if _XH_DEPS_HASH:
    XH["deps_hash"] = _XH_DEPS_HASH

# Diretório exclusivo dos artefatos locais desta auditoria. Não é o banco Turso
# nem o diretório de arquivos finais do projeto.
_AUDIT_TMP_DEFAULT = (
    "/kaggle/working/xh-audit-tmp"
    if Path("/kaggle/working").exists()
    else "/tmp/xh-audit-tmp"
)
TMP = Path(os.environ.get("XH_AUDIT_TMP_DIR", _AUDIT_TMP_DEFAULT))
TMP.mkdir(parents=True, exist_ok=True)
XH["audit_tmp_dir"] = str(TMP)

def rec(nome, status, msg=""):
    """Registra e imprime o resultado de um teste. status: PASSOU | FALHOU | PENDENTE | AVISO"""
    XH["res"][nome] = (status, msg)
    print(f"[{status}] {nome}" + (f" — {msg}" if msg else ""))

def precisa_celula1():
    if not XH.get("root"):
        raise SystemExit("Rode a célula de código 1 primeiro (e ela precisa terminar com PASSOU).")

def achar_raiz(base):
    """Raiz REAL do projeto = pasta que tem requirements-kaggle.txt e xhakinha/cloud/brain.py (não assume estrutura)."""
    for c in (base, base / "xhakinha-clips-ai", base / "xhakinha-clips-ai" / "xhakinha-clips-ai"):
        if (c / "requirements-kaggle.txt").is_file() and (c / "xhakinha" / "cloud" / "brain.py").is_file():
            return c
    return None

def git(*args, env=None, cwd=None, timeout=240):
    r = subprocess.run(["git", *args], capture_output=True, text=True, env=env, cwd=cwd, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()

# --- autenticação GitHub sem token na URL, sem imprimir, sem gravar em arquivo
env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{tok}".encode()).decode()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"})
    print("GITHUB_TOKEN: OK (não exibido)")
except Exception:
    print("GITHUB_TOKEN: AUSENTE — tentando sem autenticação (só funciona se o repositório for público)")

BASE.mkdir(parents=True, exist_ok=True)

# Preservar somente a alteração local conhecida em analysis.py.
# Qualquer outra alteração local bloqueia o pull; nunca usar reset/clean.
analysis_rel = "xhakinha-clips-ai/xhakinha/analysis.py"
analysis_path = CLONE / analysis_rel
analysis_backup = None
analysis_hash = None

if (CLONE / ".git").is_dir():
    rc, branch_atual = git("-C", str(CLONE), "rev-parse", "--abbrev-ref", "HEAD")
    if rc != 0 or branch_atual != BRANCH:
        rec("1. Repositório", "FALHOU",
            f"branch atual={branch_atual or 'indisponível'}; esperado={BRANCH}")
        raise SystemExit("Branch inesperada. Sincronização bloqueada.")

    rc, status = git("-C", str(CLONE), "status", "--porcelain", "--untracked-files=all")
    if rc != 0:
        rec("1. Repositório", "FALHOU", "não foi possível consultar git status")
        raise SystemExit("Estado Git não verificável.")

    linhas = [ln for ln in status.splitlines() if ln.strip()]
    caminhos = [ln[3:].strip() for ln in linhas]
    inesperados = [p for p in caminhos if p != analysis_rel]

    if inesperados:
        rec("1. Repositório", "FALHOU",
            "há alterações locais além de analysis.py: " + ", ".join(inesperados[:8]))
        raise SystemExit(
            "Pull bloqueado para proteger alterações locais não reconhecidas."
        )

    if analysis_rel in caminhos:
        if not analysis_path.is_file():
            raise SystemExit("analysis.py aparece modificado, mas não foi encontrado.")
        import shutil
        analysis_backup = Path("/kaggle/working") / (
            f"analysis.py.preservado-{time.time_ns()}"
        )
        shutil.copy2(analysis_path, analysis_backup)
        analysis_hash = hashlib.sha256(analysis_backup.read_bytes()).hexdigest()

        # Restaurar temporariamente apenas esse arquivo ao conteúdo do HEAD,
        # para permitir fast-forward; a cópia local fica preservada fora do clone.
        show = subprocess.run(
            ["git", "-C", str(CLONE), "show", f"HEAD:{analysis_rel}"],
            capture_output=True
        )
        if show.returncode != 0:
            shutil.copy2(analysis_backup, analysis_path)
            raise SystemExit("Não foi possível obter analysis.py do HEAD; backup preservado.")
        analysis_path.write_bytes(show.stdout)

    rc, out = git("-C", str(CLONE), "pull", "--ff-only", "origin", BRANCH, env=env)
    print("git pull:", "OK" if rc == 0 else "FALHOU — " + out[-200:])
    if rc != 0:
        if analysis_backup and analysis_backup.is_file():
            import shutil
            shutil.copy2(analysis_backup, analysis_path)
        rec("1. Repositório", "FALHOU",
            "git pull --ff-only falhou; arquivo local analysis.py foi preservado.")
        raise SystemExit("Sincronização do GitHub falhou. Não continue.")

    if analysis_backup and analysis_backup.is_file():
        import shutil
        shutil.copy2(analysis_backup, analysis_path)
        hash_restaurado = hashlib.sha256(analysis_path.read_bytes()).hexdigest()
        if hash_restaurado != analysis_hash:
            raise RuntimeError("SHA256 de analysis.py divergiu após restauração.")
        XH["analysis_backup"] = str(analysis_backup)
        XH["analysis_sha256"] = analysis_hash
        rec("1. analysis.py", "PASSOU",
            f"cópia preservada e SHA256 conferido: {analysis_hash[:16]}…")
    else:
        rec("1. analysis.py", "PASSOU",
            "nenhuma alteração local em analysis.py precisou ser preservada.")

elif CLONE.exists():
    rec("1. Repositório", "FALHOU",
        f"{CLONE} existe, mas não é um clone Git. Nada foi alterado.")
    raise SystemExit
else:
    rc, out = git("clone", "--depth", "1", "--branch", BRANCH, REPO, str(CLONE), env=env)
    if rc != 0:
        msg = out[-200:]
        if "tok" in globals() and tok:
            msg = msg.replace(tok, "***")
        rec("1. Repositório", "FALHOU", "clone falhou: " + msg)
        XH["repo_sync_ok"] = False
        raise SystemExit

ROOT = achar_raiz(CLONE)
if ROOT is None:
    rec("1. Repositório", "FALHOU",
        "não achei requirements-kaggle.txt + xhakinha/cloud/brain.py dentro do clone")
    raise SystemExit

XH["root"] = ROOT
_, br = git("-C", str(CLONE), "rev-parse", "--abbrev-ref", "HEAD")
_, sha = git("-C", str(CLONE), "rev-parse", "--short", "HEAD")
print("RAIZ DO PROJETO:", ROOT)
print("BRANCH/COMMIT  :", br, sha)
XH["repo_sync_ok"] = (br == BRANCH and bool(sha))
rec("1. Repositório", "PASSOU" if XH["repo_sync_ok"] else "FALHOU",
    f"branch={br} commit={sha}")
if not XH["repo_sync_ok"]:
    raise SystemExit("O clone não está na branch main. Não continue.")

# --- dependências: UMA vez por kernel (e só se o requirements mudou)
req = ROOT / "requirements-kaggle.txt"
h = hashlib.sha256(req.read_bytes()).hexdigest()
if XH.get("deps_hash") == h:
    print("Dependências: já instaladas neste kernel (pulando).")
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], capture_output=True, text=True)
    if r.returncode == 0:
        XH["deps_hash"] = h
        print("Dependências: instaladas.")
    else:
        print("pip FALHOU:", r.stderr[-400:])
XH["deps_ok"] = (XH.get("deps_hash") == h)
rec("1. Dependências", "PASSOU" if XH["deps_ok"] else "FALHOU", "requirements-kaggle.txt")
if not XH["deps_ok"]:
    raise SystemExit("Dependências não foram instaladas corretamente. Não continue.")

# --- sys.path: só a raiz real (remove caminhos duplicados/errados do clone)
sp = [p for p in sys.path if not str(p).startswith(str(BASE)) or str(p) == str(ROOT)]
sys.path[:] = sp
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
for m in [m for m in sys.modules if m == "xhakinha" or m.startswith("xhakinha.")]:
    f = getattr(sys.modules[m], "__file__", "") or ""
    if f and not f.startswith(str(ROOT)):
        print("AVISO: módulo xhakinha já importado de OUTRO caminho:", f, "→ use Run > Restart session e rode a célula de código 1 de novo.")
        break

try:
    import libsql
    XH["tem_turso"] = True
except Exception:
    XH["tem_turso"] = False
print("Driver libsql disponível:", "SIM" if XH["tem_turso"] else "NÃO")

# --- helpers de coordenador usados por várias células (somente leitura)
def coord():
    from xhakinha.cloud.client import HttpClient
    return HttpClient(os.environ["XH_COORD_URL"], os.environ["XH_COORD_TOKEN"], timeout=20)

def mostrar_estado(st, titulo="ESTADO DO COORDENADOR"):
    now = st.get("now", time.time())
    print(f"--- {titulo} ---")
    print("tarefas por status:", st.get("counts"))
    for l in st.get("locks", []):
        print(f"lock '{l['name']}': dono={l['owner']} fence={l['fence']} lease restante={round(l['lease_until'] - now)}s")
    if not st.get("locks"):
        print("locks: nenhum")
    ex = st.get("executors", [])
    print(f"executores ({len(ex)}):")
    for e in ex:
        print(f"  {e['id']:<34} {e['status']:<9} visto há {e.get('seen_ago_s')}s  restante={e.get('time_left_s')}s")
    tk = st.get("tasks", [])
    print("tarefas abertas:", len(tk), "|", {t['owner'] or '-': 0 for t in tk} if tk else "nenhuma")
    for ev in st.get("events", [])[:5]:
        print("  evento:", ev.get("level"), ev.get("component"), str(ev.get("message"))[:100])
print("Célula de código 1 concluída.")


## 2) Secrets — somente lê (imprime só OK/AUSENTE; nunca o valor)

In [ ]:
precisa_celula1()

# ============================================================
# ETAPA 2 — SECRETS E CONFIGURAÇÃO DOS BACKENDS
# Armazenamento: Backblaze B2
# Estado canônico: Turso
# Sem conexão, migração ou escrita no banco nesta etapa.
# ============================================================

OBRIGATORIOS = [
    "SECRET_KEY",
    "XH_COORD_URL",
    "XH_COORD_TOKEN",
    "TURSO_DATABASE_URL",
    "TURSO_AUTH_TOKEN",
    "B2_ENDPOINT_URL",
    "B2_REGION",
    "B2_KEY_ID",
    "B2_APPLICATION_KEY",
    "B2_BUCKET",
]

OPCIONAIS = [
    "YOUTUBE_API_KEY",
    "TWITCH_CLIENT_ID",
    "TWITCH_CLIENT_SECRET",
    "TIKTOK_CLIENT_KEY",
    "TIKTOK_CLIENT_SECRET",
    "PUBLIC_BASE_URL",
]

try:
    from kaggle_secrets import UserSecretsClient
    cli = UserSecretsClient()
except Exception:
    cli = None

faltam = []

for nome in OBRIGATORIOS + OPCIONAIS:
    if not os.environ.get(nome) and cli:
        try:
            valor = cli.get_secret(nome)
            if valor:
                os.environ[nome] = valor
        except Exception:
            pass

    presente = bool(os.environ.get(nome))
    tipo = "obrigatório" if nome in OBRIGATORIOS else "opcional"

    print(
        f"{nome:<24} "
        f"{'OK' if presente else 'AUSENTE':<8} ({tipo})"
    )

    if nome in OBRIGATORIOS and not presente:
        faltam.append(nome)

# Configuração explícita dos serviços.
os.environ["DATA_DIR"] = os.environ.get(
    "DATA_DIR",
    "/kaggle/working/xh-data"
)
os.environ["XH_ENV"] = "production"
os.environ["COOKIE_SECURE"] = "1"
os.environ["XH_STATE_BACKEND"] = "turso"
os.environ["XH_STORAGE_BACKEND"] = "b2"

from xhakinha.config import load_settings

XH["backend"] = "indeterminado"
XH["turso_config_ok"] = False

# Não instanciar Store nesta etapa:
# Store.__init__ chama _init(), que pode criar ou migrar tabelas.
try:
    settings_secret = load_settings(validate=True)

    driver_ok = bool(XH.get("tem_turso"))
    credenciais_ok = bool(
        settings_secret.turso_url
        and settings_secret.turso_token
    )
    remoto_configurado = bool(
        settings_secret.state_backend == "turso"
        and settings_secret.use_turso
    )

    b2_ok = all(
        os.environ.get(nome)
        for nome in (
            "B2_ENDPOINT_URL",
            "B2_REGION",
            "B2_KEY_ID",
            "B2_APPLICATION_KEY",
            "B2_BUCKET",
        )
    )

    turso_config_ok = (
        driver_ok
        and credenciais_ok
        and remoto_configurado
    )

    XH["backend"] = (
        "turso" if remoto_configurado else "sqlite"
    )
    XH["turso_config_ok"] = turso_config_ok

    # A conexão real será validada na etapa própria.
    XH["turso_ready"] = False

    print("Driver libsql:", driver_ok)
    print("Credenciais Turso:", credenciais_ok)
    print("Turso selecionado:", remoto_configurado)
    print("Credenciais B2:", b2_ok)
    print("Armazenamento selecionado:",
          os.environ.get("XH_STORAGE_BACKEND"))

except Exception as exc:
    XH["settings_error"] = (
        f"{type(exc).__name__}: {str(exc)[:160]}"
    )
    print("Falha na configuração:", XH["settings_error"])
    turso_config_ok = False
    b2_ok = False

rec(
    "2. Secrets obrigatórios",
    "FALHOU" if faltam else "PASSOU",
    ("faltam: " + ", ".join(faltam))
    if faltam else "todos presentes"
)

rec(
    "2. Secrets TikTok",
    "PASSOU"
    if (
        os.environ.get("TIKTOK_CLIENT_KEY")
        and os.environ.get("TIKTOK_CLIENT_SECRET")
    )
    else "PENDENTE",
    "presença das duas credenciais; nenhuma chamada OAuth foi feita"
)

rec(
    "2. Driver libsql",
    "PASSOU" if XH.get("tem_turso") else "FALHOU",
    "driver instalado e importável"
    if XH.get("tem_turso")
    else "libsql indisponível"
)

rec(
    "2. Configuração Turso",
    "PASSOU" if turso_config_ok else "FALHOU",
    f"backend={XH.get('backend')}; configuração validada; conexão pendente"
)

rec(
    "2. Configuração B2",
    "PASSOU" if b2_ok else "FALHOU",
    "cinco credenciais presentes"
    if b2_ok else "credenciais B2 incompletas"
)

if faltam or not turso_config_ok or not b2_ok:
    raise SystemExit(
        "Secrets essenciais ou configuração dos backends não aprovados."
    )

print("\nETAPA 2: CONFIGURAÇÃO APROVADA.")
print("Turso: conexão real ainda precisa ser validada.")
print("B2: credenciais carregadas; acesso ao bucket ainda precisa ser validado.")


## 3) Healthcheck local — teste (ffmpeg, ffprobe, Whisper, rostos, GPU, disco)

In [ ]:
precisa_celula1()
from xhakinha.cloud.healthcheck import local_report
rep = local_report()
print(json.dumps(rep, indent=2))
rec("3. Healthcheck", "PASSOU" if rep.get("ok") else "FALHOU", f"ffmpeg={rep.get('ffmpeg')} ffprobe={rep.get('ffprobe')} gpu={rep.get('gpu')} disco_livre={rep.get('disk_free_gb')}GB")

## 4) Imports e dependências — teste

In [ ]:
precisa_celula1()
import importlib
mods = ["xhakinha.config", "xhakinha.db", "xhakinha.engine", "xhakinha.scheduler", "xhakinha.cloud.brain", "xhakinha.cloud.runner",
        "xhakinha.cloud.kaggle_runner", "xhakinha.cloud.snapshots", "xhakinha.cloud.b2_storage", "xhakinha.storage", "xhakinha.cloud.client",
        "xhakinha.sources.youtube", "xhakinha.sources.twitch", "xhakinha.sources.instagram_src", "xhakinha.publish.tiktok"]
bad = []
for m in mods:
    try:
        mod = importlib.import_module(m)
        f = str(getattr(mod, "__file__", ""))
        if not f.startswith(str(XH["root"])):
            bad.append(f"{m} veio de {f}")
    except Exception as e:
        bad.append(f"{m}: {type(e).__name__} {str(e)[:80]}")
for b in bad: print("  PROBLEMA:", b)
for pkg in ["requests", "starlette", "cryptography", "cv2", "numpy", "faster_whisper"] + (["libsql"] if XH.get("tem_turso") else []):
    try: importlib.import_module(pkg); print("  dep OK:", pkg)
    except Exception as e: bad.append(f"dep {pkg}: {type(e).__name__}"); print("  dep FALTA:", pkg)
rec("4. Imports/dependências", "FALHOU" if bad else "PASSOU",
    "; ".join(bad)[:300] if bad else f"{len(mods)} módulos")
rec("4. Driver libsql", "PASSOU" if XH.get("tem_turso") else "FALHOU",
    "libsql importável" if XH.get("tem_turso") else "libsql ausente")


## 5) Banco SQLite local e Turso — SOMENTE LEITURA (abre com mode=ro; não cria nem altera nada)

In [ ]:
precisa_celula1()

import sqlite3
from pathlib import Path
from xhakinha.config import load_settings
from xhakinha.dbdriver import LibsqlConn

S = load_settings(validate=False)
dbp = Path(S.db_path)
XH["db_path"] = dbp

print("DATA_DIR:", S.data_dir)
print("DB SQLite local:", dbp, "| existe:", dbp.exists())

TABELAS = [
    "sources", "contents", "jobs", "clips",
    "publications", "settings", "events"
]

cnt = {}

# ------------------------------------------------------------
# SQLITE LOCAL — somente leitura, se existir.
# ------------------------------------------------------------
if dbp.exists():
    XH["db_antes"] = (dbp.stat().st_size, dbp.stat().st_mtime_ns)

    c = sqlite3.connect(
        f"file:{dbp.resolve()}?mode=ro",
        uri=True
    )
    try:
        qc = c.execute("PRAGMA quick_check").fetchone()[0]
        tabs = {
            r[0] for r in c.execute(
                "SELECT name FROM sqlite_master WHERE type='table'"
            )
        }

        for tabela in TABELAS:
            if tabela in tabs:
                cnt[tabela] = c.execute(
                    f"SELECT COUNT(*) FROM {tabela}"
                ).fetchone()[0]
    finally:
        c.close()

    print("quick_check:", qc, "| contagens:", cnt)

    if qc != "ok":
        rec("5. SQLite local", "FALHOU",
            f"quick_check={qc}")
    else:
        rec("5. SQLite local", "PASSOU",
            "banco local íntegro; leitura somente")
else:
    rec("5. SQLite local", "PASSOU",
        "arquivo local ausente; backend oficial é Turso")

XH["db_local_counts"] = cnt

# ------------------------------------------------------------
# TURSO — conexão direta, exclusivamente para leitura.
# Não instancia Store e não executa migrações.
# ------------------------------------------------------------
conn = None

XH["turso_ready"] = False

try:
    if not XH.get("turso_config_ok"):
        raise RuntimeError(
            "Configuração Turso não foi aprovada na etapa 2"
        )

    if not XH.get("tem_turso"):
        raise RuntimeError("Driver libsql indisponível")

    if not S.use_turso:
        raise RuntimeError(
            "Settings não selecionou o backend Turso"
        )

    if not S.turso_url or not S.turso_token:
        raise RuntimeError(
            "Credenciais Turso ausentes nas configurações"
        )

    conn = LibsqlConn(S.turso_url, S.turso_token)

    teste = conn.execute(
        "SELECT 1 AS connection_ok"
    ).fetchone()

    if not teste or int(teste["connection_ok"]) != 1:
        raise RuntimeError("SELECT 1 não confirmou a conexão")

    rows = conn.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type='table' ORDER BY name"
    ).fetchall()

    tabelas_turso = {r["name"] for r in rows}
    if "settings" not in tabelas_turso:
        raise RuntimeError(
            "Tabela settings não encontrada no Turso"
        )

    tcnt = {}

    for tabela in TABELAS:
        if tabela in tabelas_turso:
            resultado = conn.execute(
                f"SELECT COUNT(*) AS n FROM {tabela}"
            ).fetchone()

            tcnt[tabela] = int(resultado["n"])

    XH["turso_counts"] = tcnt
    XH["turso_sources_count"] = int(
        tcnt.get("sources", 0)
    )
    XH["turso_tables"] = sorted(tabelas_turso)
    XH["backend"] = "turso"
    XH["turso_ready"] = True

    print("Turso SELECT 1: OK")
    print("Tabelas encontradas:", len(tabelas_turso))
    print("Turso remoto:", {
        k: tcnt.get(k, 0) for k in TABELAS
    })

    rec(
        "5. Turso (leitura)",
        "PASSOU",
        f"{len(tabelas_turso)} tabelas; "
        f"sources={XH['turso_sources_count']}"
    )

except Exception as exc:
    XH["turso_counts"] = None
    XH["turso_sources_count"] = None

    rec(
        "5. Turso (leitura)",
        "FALHOU",
        f"{type(exc).__name__}: {str(exc)[:160]}"
    )

finally:
    if conn is not None:
        conn.close()
        

## 6) Coordenador — `GET /api/state` com os headers corretos (somente leitura)


In [ ]:
precisa_celula1()

# Um novo baseline deve substituir qualquer leitura antiga deste ciclo.
XH.pop("state", None)
XH["coordinator_baseline_ok"] = False
XH["coordinator_baseline_source"] = "cell6"

try:
    st = coord().state()
    ok = (
        isinstance(st, dict)
        and isinstance(st.get("now"), (int, float))
        and isinstance(st.get("counts"), dict)
        and isinstance(st.get("executors"), list)
        and isinstance(st.get("locks"), list)
    )
    if not ok:
        raise RuntimeError("Resposta /api/state incompleta ou em formato inesperado")

    XH["state"] = st
    XH["coordinator_baseline_ok"] = True
    XH["coordinator_baseline_at"] = time.time()
    mostrar_estado(st, "BASELINE DO COORDENADOR")
    rec(
        "6. Coordenador /api/state",
        "PASSOU",
        f"baseline capturado; executores={len(st['executors'])}; tarefas={st.get('counts')}"
    )
    rec(
        "6b. Turso via coordenador",
        "PENDENTE",
        "/api/state não comprova Turso; a conexão SQL é validada na célula de código 5"
    )
except Exception as exc:
    XH.pop("state", None)
    XH["coordinator_baseline_ok"] = False
    rec(
        "6. Coordenador /api/state",
        "FALHOU",
        f"{type(exc).__name__}: {str(exc)[:180]}"
    )
    rec(
        "6b. Turso via coordenador",
        "PENDENTE",
        "coordenador indisponível; Turso é validado separadamente"
    )


## 7) Backblaze B2 — autenticação, quota e listagem de snapshots (SOMENTE LEITURA: sem upload, sem apagar, sem criar pastas)


In [ ]:
precisa_celula1()

from xhakinha.cloud.runner import drive_from_env
from xhakinha.cloud.snapshots import SnapshotManager

TMP = (
    Path("/kaggle/working/xh-diag-tmp")
    if Path("/kaggle/working").exists()
    else Path("/tmp/xh-diag-tmp")
)
TMP.mkdir(parents=True, exist_ok=True)

XH["b2"] = None
XH["drive"] = None
XH["b2_read_ok"] = False
XH["snapshot_list_ok"] = False
XH["manifests"] = []

try:
    # --------------------------------------------------------
    # BACKEND: deve ser exclusivamente B2.
    # --------------------------------------------------------
    if os.environ.get("XH_STORAGE_BACKEND", "").strip().lower() != "b2":
        raise RuntimeError(
            "XH_STORAGE_BACKEND não está configurado como b2"
        )

    storage = drive_from_env(os.environ)

    # SnapshotManager e as etapas seguintes usam esta interface
    # genérica. Mantemos o alias para compatibilidade interna.
    XH["b2"] = storage
    XH["drive"] = storage

    # --------------------------------------------------------
    # QUOTA: consulta de listagem, sem modificar objetos.
    # --------------------------------------------------------
    q = storage.quota()

    gb = lambda b: (
        None if b is None else round(b / 1_000_000_000, 3)
    )

    print("BACKEND:", type(storage).__name__)
    print("BUCKET:", storage.bucket)
    print("REGIÃO:", os.environ.get("B2_REGION"))
    print("USO B2:", gb(q.get("usage")), "GB")
    print("LIMITE CONFIGURADO:", gb(q.get("limit")), "GB")

    XH["b2_quota"] = q
    XH["b2_read_ok"] = True

    rec(
        "7. B2 autenticação/quota",
        "PASSOU",
        f"bucket acessível; uso={gb(q.get('usage'))}GB"
    )

    # --------------------------------------------------------
    # SNAPSHOTS: somente listar e ler manifestos existentes.
    # Nenhum upload, delete ou criação de pasta.
    # --------------------------------------------------------
    arqs = storage.listdir("db/snapshots")

    gz = [
        f for f in arqs
        if f["path"].endswith(".sqlite.gz")
    ]

    jsons = [
        f for f in arqs
        if f["path"].endswith(".json")
    ]

    print(
        "Objetos em db/snapshots:",
        len(arqs),
        "| .sqlite.gz:",
        len(gz),
        "| manifestos:",
        len(jsons)
    )

    sm = SnapshotManager(storage, TMP / "list")
    manifests = sm.manifests()

    XH["manifests"] = manifests

    for m in manifests[:5]:
        print(
            "manifesto:",
            m.get("file"),
            "| epoch:",
            m.get("epoch"),
            "| ts:",
            m.get("ts"),
            "| contagens:",
            m.get("counts")
        )

    # --------------------------------------------------------
    # RECUPERAÇÃO: exige manifesto válido para permitir restore.
    # --------------------------------------------------------
    if manifests:
        arquivos_manifestados = {
            m.get("file") for m in manifests
        }

        snapshots_sem_manifesto = [
            f["path"] for f in gz
            if f["path"] not in arquivos_manifestados
        ]

        if snapshots_sem_manifesto:
            print(
                "ATENÇÃO — snapshots sem manifesto:",
                snapshots_sem_manifesto[:10]
            )

        XH["snapshot_list_ok"] = True

        rec(
            "7. B2 snapshots",
            "PASSOU",
            f"{len(manifests)} manifesto(s); "
            f"mais recente={manifests[0].get('file')}"
        )

    elif gz:
        XH["snapshot_list_ok"] = False

        rec(
            "7. B2 snapshots",
            "FALHOU",
            "há arquivos .sqlite.gz, mas nenhum manifesto válido; "
            "não iniciar vazio"
        )

    else:
        XH["snapshot_list_ok"] = False

        rec(
            "7. B2 snapshots",
            "FALHOU",
            "nenhum snapshot encontrado no B2; "
            "recuperação do estado ainda não comprovada"
        )

except Exception as e:
    XH["b2_read_ok"] = False
    XH["snapshot_list_ok"] = False

    detalhe = f"{type(e).__name__}: {str(e)[:150]}"

    rec(
        "7. B2 autenticação/quota",
        "FALHOU",
        detalhe
    )

    rec(
        "7. B2 snapshots",
        "FALHOU",
        "B2 não validado; listagem de snapshots indisponível"
    )

    print("ERRO NA ETAPA 7:", detalhe)
    

## 8) Lock do Brain e executores — diagnóstico (~20 s; NÃO adquire lock, NÃO registra executor)

In [ ]:
precisa_celula1()
try:
    c8 = coord()
    s1 = c8.state(); time.sleep(20); s2 = c8.state()
    if not all(isinstance(x, dict) and "now" in x and isinstance(x.get("locks"), list) for x in (s1, s2)):
        raise RuntimeError("formato de lock inesperado no /api/state")
    now = s2["now"]
    mostrar_estado(s2, "ESTADO ATUAL")
    L1 = {l.get("name"): l for l in s1.get("locks", [])}.get("brain")
    L2 = {l.get("name"): l for l in s2.get("locks", [])}.get("brain")
    ativo1 = bool(L1 and float(L1.get("lease_until", 0)) > float(s1["now"]))
    ativo2 = bool(L2 and float(L2.get("lease_until", 0)) > float(now))
    if ativo1 or ativo2:
        owner = (L2 or L1 or {}).get("owner", "desconhecido")
        veredito = f"BLOQUEADO: lock brain válido observado; dono={owner}. Não inicie outro Brain."
        cor = "FALHOU"
    elif L1 or L2:
        mesmo_lock_expirado = bool(
            L1 and L2
            and L1.get("owner") == L2.get("owner")
            and L1.get("fence") == L2.get("fence")
            and float(L2.get("lease_until", 0)) <= float(now)
            and float(L2.get("lease_until", 0)) <= float(L1.get("lease_until", 0))
        )
        if mesmo_lock_expirado:
            veredito = f"PASSOU: lock expirado e sem renovação nas duas leituras; dono antigo={L2.get('owner')} fence={L2.get('fence')}."
            cor = "PASSOU"
            XH["expired_lock"] = {"owner": L2.get("owner"), "fence": L2.get("fence")}
        else:
            veredito = "BLOQUEADO: lock expirado/instável ou mudou entre leituras; repita a checagem."
            cor = "AVISO"
    else:
        veredito = "PASSOU: lock brain ausente nas duas leituras separadas por 20 s."
        cor = "PASSOU"
        XH["expired_lock"] = None
    XH["lock_ok"] = (cor == "PASSOU")
    XH["lock_info"] = veredito
    print("VEREDITO:", veredito)
    ativos = [e for e in s2.get("executors", []) if e.get("status") != "stopped"]
    orfaos = [e.get("id") for e in ativos if (e.get("seen_ago_s") or 0) > 300]
    print("executores ativos:", len(ativos), "| sem heartbeat >300s:", orfaos or "nenhum")
    print("Lembrete: executor não equivale a Brain; não remova registros para liberar lock.")
    rec("8. Lock do Brain", cor, veredito)
except Exception as e:
    XH["lock_ok"] = False
    rec("8. Lock do Brain", "FALHOU", f"{type(e).__name__}: {str(e)[:150]}")

## 9) Snapshot restaurado em CÓPIA ISOLADA — teste (nunca toca no banco oficial)

In [ ]:
precisa_celula1()

# A célula de código 9 não depende da variável TMP criada pela listagem B2.
TMP = Path(XH.get("audit_tmp_dir", "/tmp/xh-audit-tmp"))
TMP.mkdir(parents=True, exist_ok=True)

import sqlite3
import uuid

from xhakinha.config import load_settings
from xhakinha.db import Store, SCHEMA_VERSION
from xhakinha.dbdriver import LibsqlConn

XH["restore_ok"] = False
XH["restored_copy"] = None
XH["turso_export_ok"] = False

if XH.get("turso_ready") is not True:
    rec(
        "9. Restore em cópia",
        "PENDENTE",
        "execute primeiro a célula de código 5 e aprove a leitura do Turso"
    )
    raise SystemExit("Restore adiado: Turso ainda não foi validado na célula de código 5.")

con = None
store = None

try:
    # 1. Confirmar Turso e conferir a versão do esquema sem escrever.
    S = load_settings(validate=False)

    if not S.use_turso:
        raise RuntimeError("Backend não está configurado para Turso.")

    con = LibsqlConn(S.turso_url, S.turso_token)

    versao = con.execute(
        "SELECT value FROM settings WHERE key='schema_version'"
    ).fetchone()

    if not versao:
        raise RuntimeError("schema_version ausente no Turso.")

    versao_atual = int(json.loads(versao["value"]))

    if versao_atual != SCHEMA_VERSION:
        raise RuntimeError(
            f"Esquema Turso={versao_atual}; "
            f"código exige={SCHEMA_VERSION}. "
            "Não será feita migração automática."
        )

    tabelas = {
        r["name"] for r in con.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
    }

    obrigatorias = {"sources", "contents", "jobs", "clips", "publications", "settings"}

    if not obrigatorias.issubset(tabelas):
        raise RuntimeError(
            f"Tabelas ausentes no Turso: {sorted(obrigatorias - tabelas)}"
        )

    tabelas_contagem = [
        "sources", "contents", "jobs",
        "clips", "publications", "settings", "events"
    ]

    contagens_remotas = {}

    for tabela in tabelas_contagem:
        if tabela in tabelas:
            contagens_remotas[tabela] = int(
                con.execute(
                    f"SELECT COUNT(*) AS n FROM {tabela}"
                ).fetchone()["n"]
            )

    print("Turso conectado: OK")
    print("Versão do esquema:", versao_atual)
    print("Tabelas:", len(tabelas))
    print("Contagens remotas:", contagens_remotas)

    # 2. Store só é instanciado após confirmar esquema compatível.
    store = Store(S)

    if store.backend != "turso":
        raise RuntimeError(
            f"Store selecionou backend inesperado: {store.backend}"
        )

    # 3. Exportar para arquivo novo e exclusivo.
    copia = (
        TMP / "turso-export" /
        f"xhakinha-{uuid.uuid4().hex[:12]}.sqlite"
    )

    copia.parent.mkdir(parents=True, exist_ok=True)

    if copia.exists():
        raise RuntimeError("Arquivo de exportação já existe.")

    store.export_sqlite(str(copia))

    if not copia.is_file() or copia.stat().st_size == 0:
        raise RuntimeError("Exportação não produziu banco SQLite.")

    # 4. Validar a cópia local em modo somente leitura.
    c = sqlite3.connect(
        f"file:{copia.resolve()}?mode=ro",
        uri=True
    )

    try:
        qc = c.execute("PRAGMA quick_check").fetchone()[0]

        tabelas_locais = {
            r[0] for r in c.execute(
                "SELECT name FROM sqlite_master WHERE type='table'"
            )
        }

        if qc != "ok":
            raise RuntimeError(f"quick_check falhou: {qc}")

        if not obrigatorias.issubset(tabelas_locais):
            raise RuntimeError("Cópia exportada possui tabelas ausentes.")

        contagens_locais = {}

        for tabela in tabelas_contagem:
            if tabela in tabelas_locais:
                contagens_locais[tabela] = int(
                    c.execute(
                        f"SELECT COUNT(*) FROM {tabela}"
                    ).fetchone()[0]
                )

    finally:
        c.close()

    # 5. Conferir que a exportação preservou as contagens.
    if contagens_locais != contagens_remotas:
        raise RuntimeError(
            "Contagens diferentes entre Turso e cópia SQLite: "
            f"remoto={contagens_remotas}; local={contagens_locais}"
        )

    XH["restored_copy"] = copia
    XH["restored_counts"] = contagens_locais
    XH["restore_ok"] = True
    XH["turso_export_ok"] = True

    print("\nCÓPIA SQLITE DO TURSO: APROVADA")
    print("Arquivo:", copia)
    print("Tamanho:", copia.stat().st_size, "bytes")
    print("Quick check:", qc)
    print("Contagens conferidas:", contagens_locais)

    rec(
        "9. Restore em cópia",
        "PASSOU",
        "exportação do Turso validada em SQLite isolado; "
        "nenhuma restauração sobre o banco oficial"
    )

except Exception as e:
    XH["restore_ok"] = False
    XH["turso_export_ok"] = False

    rec(
        "9. Restore em cópia",
        "FALHOU",
        f"{type(e).__name__}: {str(e)[:200]}"
    )

finally:
    if store is not None:
        try:
            store.conn.close()
        except Exception:
            pass

    if con is not None:
        con.close()
        


## 10) Fontes e scheduler — dry-run em cópia isolada

O teste usa uma cópia do snapshot restaurado e força temporariamente `XH_STATE_BACKEND=sqlite`. Executa dois `tick()` nessa cópia, sem iniciar threads e sem escrever no Turso. A ausência de fontes cadastradas é informativa: não comprova ingestão, mas não bloqueia os diagnósticos de infraestrutura.


In [ ]:
precisa_celula1()

import sqlite3
import shutil
import uuid

from xhakinha.config import load_settings
from xhakinha.engine import Engine

TMP = Path(XH.get("audit_tmp_dir", "/tmp/xh-audit-tmp"))
TMP.mkdir(parents=True, exist_ok=True)

XH["sources_ready"] = False
XH["sources_check_ok"] = False
XH["scheduler_dryrun_ok"] = False
XH["idempotency_dryrun_ok"] = False
XH["dry_db_path"] = None

origem = XH.get("restored_copy")
restore_result = XH.get("res", {}).get("9. Restore em cópia", ("PENDENTE", "não executado"))

if XH.get("restore_ok") is not True or not origem or not Path(origem).is_file():
    detalhe = "execute primeiro a célula de código 9; a exportação/validação do Turso ainda não está disponível"
    rec("10. Fontes/scheduler", "PENDENTE", detalhe)
    rec("10. Consulta de fontes", "PENDENTE", "aguardando a cópia isolada criada na célula de código 9")
    rec("10. Fontes cadastradas", "PENDENTE", "aguardando a cópia isolada criada na célula de código 9")
    rec("11. Idempotência", "PENDENTE", "aguardando o dry-run da célula de código 10")
    print("[PENDENTE] Etapa 10 não executada: falta uma cópia Turso validada.")
else:
    backend_original = os.environ.get("XH_STATE_BACKEND")
    storage_original = os.environ.get("XH_STORAGE_BACKEND")
    data_original = os.environ.get("DATA_DIR")
    env_original = os.environ.get("XH_ENV")
    eng_dry = None
    fonte_count = 0
    ativos = []
    dry_dir = None

    try:
        origem = Path(origem).resolve()
        if not origem.is_file():
            raise FileNotFoundError(f"Cópia exportada não existe: {origem}")

        # 1. Ler a cópia exportada em modo somente leitura.
        c = sqlite3.connect(f"file:{origem}?mode=ro", uri=True)
        try:
            qc_origem = c.execute("PRAGMA quick_check").fetchone()[0]
            if qc_origem != "ok":
                raise RuntimeError(f"Cópia de origem inválida: quick_check={qc_origem}")

            cols = {r[1] for r in c.execute("PRAGMA table_info(sources)").fetchall()}
            obrigatorias = {"platform", "name", "enabled", "authorized"}
            if not obrigatorias.issubset(cols):
                raise RuntimeError(
                    "Colunas ausentes na tabela sources: "
                    + ", ".join(sorted(obrigatorias - cols))
                )
            campos = ["platform", "name", "enabled", "authorized"]
            if "interval_s" in cols:
                campos.append("interval_s")
            sql = "SELECT " + ", ".join(campos) + " FROM sources"
            fontes = c.execute(sql).fetchall()
            XH["sources_check_ok"] = True
        finally:
            c.close()

        fonte_count = len(fontes)
        # Índices estáveis: enabled=2 e authorized=3, independentemente de interval_s.
        ativos = [f for f in fontes if bool(f[2]) and bool(f[3])]
        XH["sources_ready"] = bool(ativos)
        print("Quick check da origem:", qc_origem)
        print("Fontes cadastradas:", fonte_count)
        print("Fontes habilitadas e autorizadas:", len(ativos))
        for fonte in fontes[:20]:
            print(" ", fonte)

        # Consultar sources com sucesso é requisito técnico. Ter fontes já
        # cadastradas é opcional para colocar o Worker online: o painel pode
        # incluir as fontes depois. Sem fonte autorizada, não haverá ingestão.
        rec(
            "10. Consulta de fontes",
            "PASSOU",
            f"tabela consultada; cadastradas={fonte_count}; habilitadas/autorizadas={len(ativos)}"
        )
        if ativos:
            rec(
                "10. Fontes cadastradas", "PASSOU",
                f"{fonte_count} cadastradas; {len(ativos)} habilitadas e autorizadas"
            )
        else:
            rec(
                "10. Fontes cadastradas", "PENDENTE",
                "nenhuma fonte habilitada/autorizada; adicione e autorize no painel. Worker pode iniciar ocioso."
            )

        # 2. Gerar um clone exclusivo. O original exportado não é aberto para escrita.
        dry_dir = TMP / "dry" / f"exec-{uuid.uuid4().hex[:12]}" / "xh-data"
        dry_dir.mkdir(parents=True, exist_ok=False)
        dry_db = dry_dir / "xhakinha.db"
        shutil.copy2(origem, dry_db)
        if dry_db.resolve() == origem.resolve():
            raise RuntimeError("Proteção violada: dry-run apontou para a origem.")

        # 3. Forçar SQLite + storage local + ambiente de teste SOMENTE durante este bloco.
        os.environ["XH_STATE_BACKEND"] = "sqlite"
        os.environ["XH_STORAGE_BACKEND"] = "local"
        os.environ["DATA_DIR"] = str(dry_dir)
        os.environ["XH_ENV"] = "test"

        s_dry = load_settings(validate=False)
        s_dry.data_dir = dry_dir
        eng_dry = Engine(s_dry)

        if eng_dry.db.backend != "sqlite":
            raise RuntimeError(f"Dry-run selecionou backend inesperado: {eng_dry.db.backend}")
        if Path(s_dry.db_path).resolve() != dry_db.resolve():
            raise RuntimeError(
                "Engine não está usando a cópia isolada: "
                f"esperado={dry_db}; encontrado={s_dry.db_path}"
            )

        print("Backend do Engine:", eng_dry.db.backend)
        print("Banco usado pelo dry-run:", dry_db)

        # 4. Executar ciclo diário e planejamento no clone isolado.
        import time
        from xhakinha.scheduler import day_key

        dry_now = time.time()
        dia_dry = day_key(s_dry.tz, dry_now)

        # Estado controlado somente na cópia SQLite.
        eng_dry.db.set("engine_state", "running")
        eng_dry.db.set("last_cycle_day", "__xh_dryrun_force_cycle__")

        jobs_before = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM jobs ORDER BY rowid")
        ]
        pubs_before = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM publications ORDER BY id")
        ]

        eng_dry.tick(now=dry_now)

        jobs_after_1 = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM jobs ORDER BY rowid")
        ]
        pubs_after_1 = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM publications ORDER BY id")
        ]
        ciclo_diario_executado = (
            eng_dry.db.get("last_cycle_day") == dia_dry
        )
        estado_engine_ok = eng_dry.state() == "running"

        eng_dry.tick(now=dry_now)

        jobs_after_2 = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM jobs ORDER BY rowid")
        ]
        pubs_after_2 = [
            tuple(r) for r in eng_dry.db.q("SELECT * FROM publications ORDER BY id")
        ]

        grouped = eng_dry.db.q(
            "SELECT type, status, COUNT(*) AS n FROM jobs GROUP BY type, status"
        )
        check = eng_dry.db.one("PRAGMA quick_check")
        quick_check = next(iter(check.values())) if check else None

        sem_mudanca_segundo_tick = (
            jobs_after_1 == jobs_after_2
            and pubs_after_1 == pubs_after_2
        )

        XH["scheduler_dryrun_ok"] = bool(
            quick_check == "ok"
            and ciclo_diario_executado
            and estado_engine_ok
        )
        XH["idempotency_dryrun_ok"] = bool(
            XH["scheduler_dryrun_ok"] and sem_mudanca_segundo_tick
        )

        print("\\n--- RESULTADO DO DRY-RUN ---")
        print("Jobs antes:", len(jobs_before))
        print("Jobs após primeiro tick:", len(jobs_after_1))
        print("Jobs após segundo tick:", len(jobs_after_2))
        print("Ciclo diário executado:", ciclo_diario_executado)
        print("Engine em execução:", estado_engine_ok)
        print("Publicações após primeiro tick:", len(pubs_after_1))
        print("Publicações após segundo tick:", len(pubs_after_2))
        print("Jobs agrupados:", grouped)
        print("Quick check final:", quick_check)
        print("Segundo tick sem alterações:", sem_mudanca_segundo_tick)

        rec(
            "10. Fontes/scheduler",
            "PASSOU" if XH["scheduler_dryrun_ok"] else "FALHOU",
            f"ciclo diário executado no clone SQLite; quick_check={quick_check}"
        )
        rec(
            "11. Idempotência",
            "PASSOU" if XH["idempotency_dryrun_ok"] else "FALHOU",
            "segundo tick não alterou jobs nem publicações"
            if XH["idempotency_dryrun_ok"]
            else "execução ou idempotência do dry-run não comprovada"
        )

        XH["dry_db_path"] = str(dry_db)
        XH["dry_jobs"] = grouped
        XH["dry_source_count"] = fonte_count
        XH["dry_active_source_count"] = len(ativos)

    except Exception as exc:
        XH["scheduler_dryrun_ok"] = False
        XH["idempotency_dryrun_ok"] = False
        XH["sources_ready"] = False
        rec(
            "10. Fontes/scheduler",
            "FALHOU",
            f"{type(exc).__name__}: {str(exc)[:220]}"
        )
        rec(
            "11. Idempotência",
            "FALHOU",
            "dry-run isolado não pôde ser concluído"
        )
        if XH.get("sources_check_ok") is not True:
            rec(
                "10. Consulta de fontes", "FALHOU",
                f"não foi possível ler a tabela sources: {type(exc).__name__}: {str(exc)[:160]}"
            )
        if "10. Fontes cadastradas" not in XH.get("res", {}):
            rec("10. Fontes cadastradas", "PENDENTE", "cadastro ainda não pôde ser conferido")
        print("ERRO NA ETAPA 10:", type(exc).__name__, str(exc)[:220])

    finally:
        # Fechar o banco do clone e restaurar todo o ambiente original.
        if eng_dry is not None:
            try:
                eng_dry.db.conn.close()
            except Exception:
                pass
        for nome, valor in (
            ("XH_STATE_BACKEND", backend_original),
            ("XH_STORAGE_BACKEND", storage_original),
            ("DATA_DIR", data_original),
            ("XH_ENV", env_original),
        ):
            if valor is None:
                os.environ.pop(nome, None)
            else:
                os.environ[nome] = valor

print("\nA cópia original do Turso e o banco remoto não foram usados para os ticks.")


## 11) Idempotência — confirma que as células anteriores não alteraram o banco oficial nem registraram executores

In [ ]:
precisa_celula1()

probs = []
pendencias = []

# 1. SQLite local: no backend Turso, a ausência é esperada.
dbp = XH.get("db_path")
if dbp and XH.get("db_antes") and Path(dbp).exists():
    depois = (Path(dbp).stat().st_size, Path(dbp).stat().st_mtime_ns)
    if depois != XH["db_antes"]:
        probs.append("banco SQLite local mudou durante os diagnósticos")
elif XH.get("backend") == "turso":
    print("SQLite local ausente: esperado no backend Turso.")
elif dbp and Path(dbp).exists() and not XH.get("db_antes"):
    pendencias.append("baseline do arquivo SQLite local não foi capturado na célula de código 5")

# 2. Garantir baseline de executores. Se a célula de código 6 não foi rodada,
#    recuperar agora por GET /api/state (somente leitura), em vez de falhar
#    com 'baseline do coordenador ausente'.
def _estado_coord_valido(st):
    return (
        isinstance(st, dict)
        and isinstance(st.get("now"), (int, float))
        and isinstance(st.get("counts"), dict)
        and isinstance(st.get("executors"), list)
        and isinstance(st.get("locks"), list)
    )

baseline = XH.get("state")
if not _estado_coord_valido(baseline):
    try:
        baseline = coord().state()
        if not _estado_coord_valido(baseline):
            raise RuntimeError("resposta /api/state incompleta")
        XH["state"] = baseline
        XH["coordinator_baseline_ok"] = True
        XH["coordinator_baseline_at"] = time.time()
        XH["coordinator_baseline_source"] = "recuperado na célula de código 11"
        rec(
            "6. Coordenador /api/state",
            "PASSOU",
            "baseline recuperado nesta célula por GET /api/state somente leitura"
        )
        rec(
            "6b. Turso via coordenador",
            "PENDENTE",
            "o estado do coordenador não substitui o SELECT SQL direto do Turso"
        )
    except Exception as exc:
        probs.append(
            f"não foi possível capturar baseline do coordenador: {type(exc).__name__}: {str(exc)[:120]}"
        )
else:
    XH["coordinator_baseline_ok"] = True

# 3. Comparar o conjunto de executores do baseline com uma leitura atual.
if _estado_coord_valido(baseline):
    try:
        antes = {
            str(e["id"]) for e in baseline["executors"]
            if isinstance(e, dict) and e.get("id") is not None
        }
        if len(antes) != len(baseline["executors"]):
            raise RuntimeError("baseline contém executor sem id válido")

        estado_agora = coord().state()
        if not _estado_coord_valido(estado_agora):
            raise RuntimeError("resposta atual /api/state incompleta")
        agora = {
            str(e["id"]) for e in estado_agora["executors"]
            if isinstance(e, dict) and e.get("id") is not None
        }
        if len(agora) != len(estado_agora["executors"]):
            raise RuntimeError("estado atual contém executor sem id válido")
        novos = agora - antes
        if novos:
            probs.append(f"executores novos apareceram durante a auditoria: {sorted(novos)}")
        else:
            print(f"Comparação de executores: OK ({len(antes)} no baseline; nenhum novo).")
    except Exception as exc:
        probs.append(f"comparação de executores falhou: {type(exc).__name__}: {str(exc)[:140]}")

# 4. Distinguir etapa não executada (PENDENTE) de teste executado e reprovado.
resultado_restore = XH.get("res", {}).get("9. Restore em cópia")
if XH.get("restore_ok") is not True:
    if resultado_restore and resultado_restore[0] == "FALHOU":
        probs.append("a exportação/validação da cópia Turso falhou na célula de código 9")
    else:
        pendencias.append("restore/exportação isolada ainda não aprovado: execute a célula de código 9")

resultado_scheduler = XH.get("res", {}).get("10. Fontes/scheduler")
if XH.get("scheduler_dryrun_ok") is not True:
    if resultado_scheduler and resultado_scheduler[0] == "FALHOU":
        probs.append("dry-run do scheduler falhou na célula de código 10")
    else:
        pendencias.append("dry-run do scheduler ainda não executado: execute a célula de código 10")

resultado_idem = XH.get("res", {}).get("11. Idempotência")
if XH.get("idempotency_dryrun_ok") is not True:
    if resultado_idem and resultado_idem[0] == "FALHOU":
        probs.append("teste de idempotência executado e reprovado na célula de código 10")
    else:
        pendencias.append("teste de idempotência ainda não aprovado: depende da célula de código 10")

# 5. Verificar se a configuração Turso foi comprovada nesta execução.
if XH.get("turso_ready") is not True:
    pendencias.append("leitura do Turso ainda não aprovada na célula de código 5")

XH["integridade_idempotencia_ok"] = bool(
    not probs and not pendencias
)

if probs:
    status = "FALHOU"
    detalhe = "; ".join(probs)
elif pendencias:
    status = "PENDENTE"
    detalhe = "Pré-requisitos ainda não executados/aprovados: " + "; ".join(pendencias)
else:
    status = "PASSOU"
    detalhe = (
        "baseline coordenador válido; nenhum executor novo; banco sem alteração inesperada; "
        "restore e dry-run idempotente aprovados"
    )

rec("11. Integridade/idempotência", status, detalhe[:900])
print("\nSTATUS DA ETAPA 11:", status)
if pendencias:
    print("PENDÊNCIAS:")
    for item in pendencias:
        print(" -", item)
if probs:
    print("FALHAS:")
    for item in probs:
        print(" -", item)


## 12) Relatório consolidado — PASSOU / FALHOU / PENDENTE

A lista obrigatória valida ambiente, Secrets, Turso, B2, coordenador, lock, exportação isolada, scheduler e idempotência. **É obrigatório haver ao menos uma fonte habilitada e autorizada** para liberar o fluxo de operação. O primeiro snapshot B2 é validado na Etapa 15B, depois da pré-validação.


In [ ]:
precisa_celula1()

OBRIGATORIOS = [
    "1. Repositório",
    "1. analysis.py",
    "1. Dependências",
    "2. Secrets obrigatórios",
    "2. Configuração Turso",
    "2. Configuração B2",
    "2. Driver libsql",
    "3. Healthcheck",
    "4. Imports/dependências",
    "5. SQLite local",
    "5. Turso (leitura)",
    "6. Coordenador /api/state",
    "7. B2 autenticação/quota",
    "8. Lock do Brain",
    "9. Restore em cópia",
    "10. Fontes/scheduler",
    "10. Consulta de fontes",
    "11. Idempotência",
    "11. Integridade/idempotência",
]

# Resultados das etapas anteriores
resultados = XH["res"]

print(f"{'TESTE':<38} {'STATUS':<9} DETALHE")
print("-" * 110)

for nome, (status, msg) in sorted(resultados.items()):
    print(f"{nome:<38} {status:<9} {msg[:110]}")

# Verifica os testes obrigatórios
faltantes = [
    n for n in OBRIGATORIOS
    if n not in resultados
]

nao_passaram = [
    n for n in OBRIGATORIOS
    if resultados.get(n, ("AUSENTE", ""))[0] != "PASSOU"
]

# O primeiro snapshot B2 ainda não é pré-requisito.
# Nenhuma outra falha é ignorada.
extras_falhos = [
    n for n, (status, _) in resultados.items()
    if status == "FALHOU"
    and n != "7. B2 snapshots"
]

XH["pronto"] = bool(
    not faltantes
    and not nao_passaram
    and not extras_falhos
    and XH.get("turso_ready") is True
    and XH.get("restore_ok") is True
    and XH.get("lock_ok") is True
    and XH.get("idempotency_dryrun_ok") is True
    and XH.get("integridade_idempotencia_ok") is True
    and XH.get("b2_read_ok") is True
    and XH.get("sources_check_ok") is True
)

print("\nPRÉ-REQUISITOS OBRIGATÓRIOS")
print("AUSENTES:", faltantes or "nenhum")
print("NÃO PASSARAM:", nao_passaram or "nenhum")
print("OUTROS FALHARAM:", extras_falhos or "nenhum")

# Snapshot B2: mostrar a situação real, sem aprovar artificialmente.
snapshot = resultados.get(
    "7. B2 snapshots",
    ("PENDENTE", "Primeiro snapshot ainda não criado.")
)

print("\nSNAPSHOT B2:", snapshot[0])
print("DETALHE:", snapshot[1])
print(
    "PLANO: primeiro snapshot será criado e validado "
    "na Etapa 15B, antes de qualquer inicialização."
)

print("\nFONTES CADASTRADAS:",
      resultados.get("10. Fontes cadastradas", ("PENDENTE", ""))[1])
print("CONSULTA DE FONTES:",
      resultados.get("10. Consulta de fontes", ("PENDENTE", ""))[1])

print(
    "\nPRONTO PARA A CÉLULA DE CÓDIGO 15:",
    "SIM" if XH["pronto"] else "NÃO — BLOQUEADO"
)

if not XH["pronto"]:
    print("A trava é fechada: qualquer pré-requisito obrigatório não aprovado impede o início.")
else:
    print("Validações consolidadas aprovadas.")
    print("A inicialização ainda depende do bootstrap 15B, conferência 15C e gate final 15D.")
    


## 13) Status — só consulta `/api/state` (seguro repetir sempre; NÃO use a célula 15 para isso)

In [ ]:
precisa_celula1()
try:
    mostrar_estado(coord().state(), "STATUS AGORA")
except Exception as e:
    print("Falha ao consultar:", type(e).__name__, str(e)[:150])

## 14) Executores — somente observação
Esta etapa é intencionalmente sem ação. Não chama `deregister`, não remove registros e não tenta encerrar processos remotos.
Use a célula 13 para consultar o estado. Não limpe executores para tentar liberar o lock do Brain.

In [ ]:
precisa_celula1()
print("ETAPA 14 DESATIVADA.")
print("Nenhum executor será deregistrado, removido ou marcado como parado por este notebook.")
print("Para investigar executores antigos, use somente a célula 13 (status) e confira o lock brain.")

## 15) Pré-validação e inicialização controlada

A etapa 15 (código 15) faz a pré-validação sem iniciar processos. Depois execute 15B (código 16), 15C (código 17) e 15D (código 18). A inicialização real fica no código 19 e permanece desativada por padrão (`INICIAR_KAGGLE_WORKER = False`).

Não altere a flag para True durante auditorias. Só inicie após todos os gates passarem e com autorização explícita.

In [ ]:
precisa_celula1()

XH["etapa15_preflight_ok"] = False

# ============================================================
# ETAPA 15 — INICIALIZAÇÃO AUTOMÁTICA PROTEGIDA
# PRÉ-VALIDAÇÃO FINAL — TURSO + B2 + COORDENADOR
#
# Esta versão valida a infraestrutura, mas ainda NÃO inicia
# Brain/Executor. O bootstrap do primeiro snapshot B2 precisa
# ser concluído antes da liberação de kaggle_runner.main().
# ============================================================

import os
import json
import builtins

# ------------------------------------------------------------
# 1. Impedir execução duplicada neste kernel
# ------------------------------------------------------------

if getattr(builtins, "__XH_MAIN_STARTED__", False):
    raise SystemExit(
        "main() já foi chamado neste kernel. "
        "Não repita; use a etapa 13 para consultar o status."
    )

# ------------------------------------------------------------
# 2. Confirmar aprovação da etapa 12
# ------------------------------------------------------------

if XH.get("pronto") is not True:
    raise SystemExit(
        "Etapa 12 não liberou a inicialização. "
        "Nenhum Brain ou Executor será iniciado."
    )

if XH.get("b2_read_ok") is not True:
    raise SystemExit(
        "B2 não foi validado na Etapa 7. Início bloqueado."
    )

if XH.get("sources_check_ok") is not True:
    raise SystemExit(
        "A tabela de fontes não foi consultada com sucesso na célula de código 10. Início bloqueado."
    )

for nome in (
    "turso_ready",
    "restore_ok",
    "lock_ok",
    "idempotency_dryrun_ok",
):
    if XH.get(nome) is not True:
        raise SystemExit(
            f"Pré-requisito crítico não aprovado: {nome}"
        )

# ------------------------------------------------------------
# 3. Revalidar backend e credenciais Turso
# ------------------------------------------------------------

backend15 = os.environ.get(
    "XH_STATE_BACKEND", ""
).strip().lower()

if backend15 != "turso":
    raise SystemExit(
        "XH_STATE_BACKEND não está em turso. "
        "Início bloqueado."
    )

if not (
    os.environ.get("TURSO_DATABASE_URL")
    and os.environ.get("TURSO_AUTH_TOKEN")
):
    raise SystemExit(
        "Credenciais Turso ausentes. Início bloqueado."
    )

# ------------------------------------------------------------
# 4. Validar conexão Turso diretamente, sem Store
#    Não executa migrações nem altera o banco oficial.
# ------------------------------------------------------------

from xhakinha.config import load_settings
from xhakinha.db import SCHEMA_VERSION
from xhakinha.dbdriver import LibsqlConn

s15 = load_settings(validate=False)

if not s15.use_turso:
    raise SystemExit(
        "Configuração não confirmou use_turso=True. "
        "Início bloqueado."
    )

if not s15.turso_url or not s15.turso_token:
    raise SystemExit(
        "URL ou token Turso ausente nas configurações."
    )

c15 = LibsqlConn(
    s15.turso_url,
    s15.turso_token
)

try:
    teste15 = c15.execute("SELECT 1").fetchone()

    if not teste15:
        raise SystemExit(
            "Turso não respondeu ao SELECT 1."
        )

    schema_row15 = c15.execute(
        "SELECT value FROM settings "
        "WHERE key='schema_version'"
    ).fetchone()

    if not schema_row15:
        raise SystemExit(
            "schema_version ausente no Turso."
        )

    schema_raw15 = schema_row15["value"]

    try:
        schema15 = int(json.loads(schema_raw15))
    except (TypeError, ValueError, json.JSONDecodeError):
        schema15 = int(schema_raw15)

    if schema15 != SCHEMA_VERSION:
        raise SystemExit(
            f"Versão de esquema incompatível: "
            f"Turso={schema15}; código={SCHEMA_VERSION}."
        )

finally:
    c15.close()

print("[PASSOU] Turso conectado.")
print("[PASSOU] SELECT 1 aprovado.")
print(
    f"[PASSOU] Schema Turso={schema15}; "
    f"esperado={SCHEMA_VERSION}."
)

# ------------------------------------------------------------
# 5. Confirmar estado atual do coordenador
# ------------------------------------------------------------

st15_state = coord().state()

if not isinstance(st15_state, dict):
    raise SystemExit(
        "Estado inválido do coordenador."
    )

now15 = st15_state.get("now")
locks15 = st15_state.get("locks")
executores15 = st15_state.get("executors")

if not isinstance(now15, (int, float)):
    raise SystemExit(
        "Relógio do coordenador ausente ou inválido."
    )

if not isinstance(locks15, list):
    raise SystemExit(
        "Locks do coordenador em formato inválido."
    )

if not isinstance(executores15, list):
    raise SystemExit(
        "Executores do coordenador em formato inválido."
    )

mostrar_estado(
    st15_state,
    "ÚLTIMA CHECAGEM ANTES DA INICIALIZAÇÃO"
)

# ------------------------------------------------------------
# 6. Validar lock brain
#    Aceitar somente ausência ou o mesmo lock expirado
#    identificado na etapa 8.
# ------------------------------------------------------------

brain_locks15 = [
    lk for lk in locks15
    if isinstance(lk, dict)
    and lk.get("name") == "brain"
]

if len(brain_locks15) > 1:
    raise SystemExit(
        "Há registros duplicados do lock brain."
    )

expired15 = XH.get("expired_lock")

if brain_locks15:
    lk15 = brain_locks15[0]

    lease15 = float(
        lk15.get("lease_until") or 0
    )

    if lease15 > float(now15):
        raise SystemExit(
            "Já existe Brain com lease ativo. "
            "Início bloqueado."
        )

    if not isinstance(expired15, dict):
        raise SystemExit(
            "Lock expirado não corresponde "
            "ao diagnóstico anterior."
        )

    if (
        lk15.get("owner") != expired15.get("owner")
        or lk15.get("fence") != expired15.get("fence")
    ):
        raise SystemExit(
            "Lock brain mudou desde a etapa 8. "
            "Repita os diagnósticos antes de iniciar."
        )

else:
    if expired15 is not None:
        raise SystemExit(
            "Lock brain desapareceu desde a etapa 8. "
            "Repita os diagnósticos antes de iniciar."
        )

print("[PASSOU] Lock brain livre e consistente.")

# ------------------------------------------------------------
# 7. Validar executores
#    Estados terminais conhecidos não bloqueiam.
#    Qualquer estado desconhecido bloqueia por segurança.
# ------------------------------------------------------------

terminais15 = {
    "lost",
    "stopped",
    "done",
    "failed",
    "offline",
}

indeterminados15 = [
    e for e in executores15
    if not isinstance(e, dict)
    or str(
        e.get("status", "")
    ).strip().lower() not in terminais15
]

if indeterminados15:
    raise SystemExit(
        "Há executores ativos ou com estado não reconhecido. "
        f"Início bloqueado: {indeterminados15}"
    )

print(
    "[PASSOU] Nenhum executor ativo "
    "ou com estado desconhecido."
)

# ------------------------------------------------------------
# 8. Mostrar resultados das etapas anteriores
# ------------------------------------------------------------

print(
    "[INFO] Fontes cadastradas:",
    XH.get("res", {}).get(
        "10. Fontes cadastradas",
        ("PENDENTE", "não verificado")
    )
)

print(
    "[INFO] Scheduler:",
    XH.get("res", {}).get(
        "10. Fontes/scheduler",
        ("PENDENTE", "não verificado")
    )
)

print(
    "[INFO] B2:",
    XH.get("res", {}).get(
        "2. Configuração B2",
        ("PENDENTE", "não verificado")
    )
)

# ------------------------------------------------------------
# 9. Snapshot B2
#    Não inventar snapshot nem aprovar recuperação inexistente.
# ------------------------------------------------------------

snapshot15 = XH.get("res", {}).get(
    "7. B2 snapshots",
    ("PENDENTE", "nenhum snapshot validado")
)

print(
    "\nSNAPSHOT B2:",
    snapshot15[0]
)

print(
    "DETALHE:",
    snapshot15[1]
)

print(
    "[PENDENTE] Bootstrap do primeiro snapshot B2."
)

# ------------------------------------------------------------
# 10. GPU: informação, sem presumir disponibilidade
# ------------------------------------------------------------

health15 = XH.get("res", {}).get(
    "3. Healthcheck",
    ("PENDENTE", "healthcheck não localizado")
)

print(
    "\n[INFO] Healthcheck:",
    health15
)

print(
    "[INFO] A inicialização poderá usar CPU "
    "se a GPU não estiver disponível e o runner "
    "estiver configurado para fallback."
)

# ------------------------------------------------------------
# 11. Resultado desta pré-validação
# ------------------------------------------------------------

XH["etapa15_preflight_ok"] = True
XH["etapa15_main_liberado"] = False
rec("15. Pré-validação final", "PASSOU", "Turso, lock e executores conferidos; Worker ainda bloqueado")

print("\n" + "=" * 65)
print("[PASSOU] Pré-validação final do Turso.")
print("[PASSOU] Lock brain consistente.")
print("[PASSOU] Executores em estados terminais conhecidos.")
print("[PASSOU] Pré-requisitos das etapas anteriores confirmados.")
print("=" * 65)

print("\n[AGUARDANDO] Bootstrap e conferência final do snapshot B2 (Etapas 15B–15D).")
print("[BLOQUEADO] kaggle_runner.main() ainda não será chamado.")
print("[BLOQUEADO] Nenhum Brain ou Executor foi iniciado.")

print(
    "\nETAPA 15 — PRÉ-VALIDAÇÃO CONCLUÍDA."
)


## CÉLULA DE CÓDIGO 16 — ETAPA 15B: Bootstrap e validação do snapshot no Backblaze B2

In [ ]:
# ============================================================
# ETAPA 15B — BOOTSTRAP / VALIDAÇÃO DO SNAPSHOT B2
# Turso = banco canônico | B2 = armazenamento de snapshots
# Pode criar um snapshot SOMENTE se não houver snapshot/manifesto.
# Não inicia Brain, Engine ou Executor; não apaga objetos.
# ============================================================

import os
import json
import gzip
import uuid
import sqlite3
import tempfile
from pathlib import Path

from xhakinha.config import load_settings
from xhakinha.db import Store, SCHEMA_VERSION
from xhakinha.dbdriver import LibsqlConn
from xhakinha.cloud.client import HttpClient
from xhakinha.cloud.runner import drive_from_env
from xhakinha.cloud.snapshots import SnapshotManager
from xhakinha.cloud.brain import Brain
from xhakinha.storage import sha256_file

precisa_celula1()
XH["snapshot_bootstrap_ok"] = False
XH["snapshot_manifest"] = None

if XH.get("etapa15_preflight_ok") is not True or XH.get("pronto") is not True:
    rec("15B. Bootstrap B2", "PENDENTE", "execute e aprove a célula de código 15 antes do bootstrap")
    raise SystemExit("Bootstrap bloqueado: pré-validação da célula de código 15 não aprovada.")
if XH.get("b2_read_ok") is not True or XH.get("sources_check_ok") is not True:
    raise RuntimeError("B2 ou consulta da tabela de fontes não foram aprovados no relatório consolidado.")

# 1. Configuração já carregada e validada na Etapa 2.
obrigatorias = (
    "SECRET_KEY", "XH_COORD_URL", "XH_COORD_TOKEN",
    "TURSO_DATABASE_URL", "TURSO_AUTH_TOKEN",
    "B2_ENDPOINT_URL", "B2_REGION", "B2_KEY_ID",
    "B2_APPLICATION_KEY", "B2_BUCKET",
)
ausentes = [n for n in obrigatorias if not os.environ.get(n)]
if ausentes:
    raise RuntimeError("Secrets obrigatórios ausentes: " + ", ".join(ausentes))

os.environ["XH_STORAGE_BACKEND"] = "b2"
os.environ["XH_STATE_BACKEND"] = "turso"
os.environ["XH_ENV"] = "production"
os.environ.setdefault("DATA_DIR", "/kaggle/working/xh-data")

settings = load_settings(validate=True)
if settings.state_backend != "turso" or not settings.use_turso:
    raise RuntimeError("Settings não confirmou Turso como banco canônico.")

client = HttpClient(
    os.environ["XH_COORD_URL"],
    os.environ["XH_COORD_TOKEN"],
    timeout=30,
)
drive = drive_from_env(os.environ)
manager = SnapshotManager(
    drive, settings.data_dir / "tmp_bootstrap_b2"
)

# Helper local: valida manifesto, objeto, hashes, schema e integridade.
def validar_snapshot_b2(meta, drive_obj):
    if not isinstance(meta, dict):
        raise RuntimeError("Manifesto B2 inválido.")
    arquivo = meta.get("file")
    if not arquivo or not str(arquivo).endswith(".sqlite.gz"):
        raise RuntimeError("Manifesto não aponta para um snapshot .sqlite.gz válido.")
    if meta.get("integrity") not in (None, "ok"):
        raise RuntimeError("Manifesto não confirma integridade.")
    if not meta.get("sha256") or not meta.get("raw_sha256"):
        raise RuntimeError("Manifesto sem hashes SHA256 completos.")

    with tempfile.TemporaryDirectory(prefix="xh-b2-verify-") as tmp:
        tmp = Path(tmp)
        gz = tmp / "snapshot.sqlite.gz"
        raw = tmp / "snapshot.sqlite"
        drive_obj.download(arquivo, gz, sha256=meta["sha256"])
        if sha256_file(gz) != meta["sha256"]:
            raise RuntimeError("SHA256 do arquivo compactado diverge.")
        with gzip.open(gz, "rb") as origem, open(raw, "wb") as destino:
            while True:
                bloco = origem.read(1024 * 1024)
                if not bloco:
                    break
                destino.write(bloco)
        if sha256_file(raw) != meta["raw_sha256"]:
            raise RuntimeError("SHA256 do SQLite extraído diverge.")
        db = sqlite3.connect(f"file:{raw.resolve()}?mode=ro", uri=True)
        try:
            integrity = db.execute("PRAGMA integrity_check").fetchone()[0]
            if integrity != "ok":
                raise RuntimeError(f"integrity_check falhou: {integrity}")
            row = db.execute(
                "SELECT value FROM settings WHERE key='schema_version'"
            ).fetchone()
            if not row:
                raise RuntimeError("Snapshot sem schema_version.")
            schema = int(json.loads(row[0]))
            if schema != SCHEMA_VERSION:
                raise RuntimeError(
                    f"Schema do snapshot={schema}; código={SCHEMA_VERSION}."
                )
            fontes = db.execute("SELECT COUNT(*) FROM sources").fetchone()[0]
        finally:
            db.close()
    # Zero fontes é válido no bootstrap: elas podem ser cadastradas no painel
    # após o Worker ficar online. A tabela sources e o schema precisam existir.
    return {"file": arquivo, "schema": schema, "sources": int(fontes),
            "sha256": meta["sha256"], "raw_sha256": meta["raw_sha256"]}

# 2. Consultar o Turso em leitura e verificar schema/fontes antes de criar snapshot.
conn_check = LibsqlConn(settings.turso_url, settings.turso_token)
try:
    ping = conn_check.execute("SELECT 1 AS ok").fetchone()
    if not ping or int(ping["ok"]) != 1:
        raise RuntimeError("Turso não respondeu ao SELECT 1.")
    schema_row = conn_check.execute(
        "SELECT value FROM settings WHERE key='schema_version'"
    ).fetchone()
    if not schema_row:
        raise RuntimeError("schema_version ausente no Turso.")
    schema_remote = int(json.loads(schema_row["value"]))
    if schema_remote != SCHEMA_VERSION:
        raise RuntimeError(
            f"Schema Turso={schema_remote}; código={SCHEMA_VERSION}; sem migração automática."
        )
    fontes_remote = int(
        conn_check.execute("SELECT COUNT(*) AS n FROM sources").fetchone()["n"]
    )
finally:
    conn_check.close()
print("Fontes cadastradas no Turso:", fontes_remote)
if fontes_remote == 0:
    print("[INFO] Nenhuma fonte ainda; bootstrap permitido. Cadastre e autorize pelo painel após o Worker ficar online.")

# 3. Descobrir snapshots dinamicamente; nunca usar nome/fence fixos.
objetos = drive.listdir("db/snapshots")
caminhos = {str(o.get("path", "")) for o in objetos if isinstance(o, dict)}
manifestos = manager.manifests()

if manifestos:
    # SnapshotManager já ordena do mais recente para o mais antigo.
    candidato = manifestos[0]
    arquivo = candidato.get("file")
    if not arquivo:
        raise RuntimeError("Manifesto listado sem campo 'file'.")
    manifesto_path = str(arquivo).replace(".sqlite.gz", ".json")
    if arquivo not in caminhos or manifesto_path not in caminhos:
        raise RuntimeError("Snapshot/manifesto listado não existe no B2.")
    meta = drive.read_json(manifesto_path)
    validacao = validar_snapshot_b2(meta, drive)
    XH["snapshot_manifest"] = meta
    print("[PASSOU] Snapshot B2 existente validado; nenhum snapshot duplicado criado.")
    print(json.dumps(validacao, indent=2, ensure_ascii=False))
else:
    sobras = [p for p in caminhos if p.endswith((".sqlite.gz", ".json"))]
    if sobras:
        raise RuntimeError(
            "Há objetos de snapshot/manifesto sem par válido; criação bloqueada: "
            + ", ".join(sorted(sobras)[:8])
        )

    # 4. Criar o primeiro snapshot sob lock exclusivo e renovável.
    owner = "bootstrap-kaggle-" + uuid.uuid4().hex[:16]
    fence = client.lock_acquire("brain", owner, ttl=900)
    if fence is None:
        raise RuntimeError("Não foi possível adquirir o lock brain para o bootstrap.")

    store = None

    def confirmar_posse():
        try:
            renovacao = client.lock_renew("brain", owner, fence, ttl=900)
            if renovacao is False:
                return False
            if isinstance(renovacao, dict):
                if renovacao.get("ok") is False or renovacao.get("renewed") is False:
                    return False
            return True
        except Exception:
            return False

    try:
        if not confirmar_posse():
            raise RuntimeError("Lock perdido antes do bootstrap.")

        # Reconsulta B2 dentro do lock para evitar snapshots duplicados.
        objetos_pos = drive.listdir("db/snapshots")
        caminhos_pos = {
            str(o.get("path", "")) for o in objetos_pos
            if isinstance(o, dict)
        }
        manifestos_pos = manager.manifests()

        if manifestos_pos:
            candidato = manifestos_pos[0]
            arquivo = candidato.get("file")
            if not arquivo:
                raise RuntimeError("Manifesto concorrente sem campo 'file'.")
            manifesto_path = str(arquivo).replace(".sqlite.gz", ".json")
            if arquivo not in caminhos_pos or manifesto_path not in caminhos_pos:
                raise RuntimeError("Snapshot/manifesto concorrente não existe no B2.")
            meta_remoto = drive.read_json(manifesto_path)
            validacao = validar_snapshot_b2(meta_remoto, drive)
            XH["snapshot_manifest"] = meta_remoto
            print("[PASSOU] Snapshot concorrente validado; nenhum duplicado criado.")
        else:
            sobras_pos = [
                p for p in caminhos_pos if p.endswith((".sqlite.gz", ".json"))
            ]
            if sobras_pos:
                raise RuntimeError(
                    "Objetos órfãos após adquirir lock; criação bloqueada: "
                    + ", ".join(sorted(sobras_pos)[:8])
                )

            store = Store(settings)
            if store.backend != "turso":
                raise RuntimeError(
                    f"Store selecionou backend inesperado: {store.backend}"
                )

            meta = manager.create(
                store.backup_source(),
                int(fence),
                still_owner=confirmar_posse,
                tables=("sources", "contents", "jobs", "clips", "publications"),
                sanitize=Brain._sanitize,
            )

            if not confirmar_posse():
                raise RuntimeError("Lock perdido após criar o snapshot.")

            arquivo = meta["file"]
            manifesto_path = str(arquivo).replace(".sqlite.gz", ".json")
            meta_remoto = drive.read_json(manifesto_path)
            if not meta_remoto or meta_remoto != meta:
                raise RuntimeError("Manifesto remoto não corresponde ao snapshot criado.")

            validacao = validar_snapshot_b2(meta_remoto, drive)
            XH["snapshot_manifest"] = meta_remoto
            print("[PASSOU] Primeiro snapshot criado, relido e validado.")

        print(json.dumps(validacao, indent=2, ensure_ascii=False))

    finally:
        if store is not None:
            try:
                store.conn.close()
            except Exception:
                pass
        try:
            client.lock_release("brain", owner, fence)
            print("[PASSOU] Lock temporário do bootstrap liberado.")
        except Exception as exc:
            print(
                "[AVISO] Liberação do lock não confirmada; a Etapa 15C bloqueará "
                f"se ainda houver lock ativo: {type(exc).__name__}: {str(exc)[:120]}"
            )

XH["snapshot_bootstrap_ok"] = True
rec(
    "15B. Bootstrap B2",
    "PASSOU",
    f"snapshot validado: {XH['snapshot_manifest'].get('file')}"
)
print("[PASSOU] Etapa 15B concluída. Nenhum Brain/Engine/Executor foi iniciado.")


## CÉLULA DE CÓDIGO 17 — ETAPA 15C: Conferência final remota (somente leitura)

In [ ]:
# ETAPA 15C — CONFERÊNCIA FINAL REMOTA (SOMENTE LEITURA)

precisa_celula1()
if XH.get("snapshot_bootstrap_ok") is not True:
    rec("15C. Conferência final remota", "PENDENTE", "execute e aprove primeiro a célula de código 16 (15B)")
    raise SystemExit("Etapa 15C bloqueada: snapshot B2 não validado na célula de código 16.")
import os
from xhakinha.cloud.client import HttpClient
from xhakinha.cloud.runner import drive_from_env
from xhakinha.cloud.snapshots import SnapshotManager

XH["etapa15c_ok"] = False
XH["etapa15c_lock_free"] = False
XH["etapa15c_no_active_executors"] = False
XH["etapa15c_no_running_tasks"] = False
XH["etapa15c_snapshot_ok"] = False

client_check = HttpClient(
    os.environ["XH_COORD_URL"],
    os.environ["XH_COORD_TOKEN"],
    timeout=30,
)
drive_check = drive_from_env(os.environ)
state = client_check.state()
if not isinstance(state, dict):
    raise RuntimeError("Resposta /api/state inválida.")
now = float(state.get("now", 0))
if now <= 0:
    raise RuntimeError("Relógio do coordenador ausente ou inválido.")
locks = state.get("locks")
executores = state.get("executors")
tarefas = state.get("tasks")
if not all(isinstance(x, list) for x in (locks, executores, tarefas)):
    raise RuntimeError(
        "Resposta /api/state sem arrays explícitos de locks, executors e tasks; "
        "não é seguro presumir que não existam tarefas em execução."
    )

brain_locks = [x for x in locks if isinstance(x, dict) and x.get("name") == "brain"]
lock_free = len(brain_locks) <= 1 and all(
    not x.get("owner") or float(x.get("lease_until") or 0) <= now
    for x in brain_locks
)

terminais = {"lost", "stopped", "done", "failed", "offline"}
ativos = [
    e for e in executores
    if not isinstance(e, dict)
    or str(e.get("status", "")).strip().lower() not in terminais
]

# Tarefas queued são normais: o Worker pode processá-las. Só tarefas já
# reivindicadas/em execução bloqueiam uma nova inicialização.
status_em_execucao = {"running", "leased", "claimed", "processing"}
tarefas_em_execucao = [
    t for t in tarefas
    if isinstance(t, dict)
    and str(t.get("status", "")).strip().lower() in status_em_execucao
]
queued = [
    t for t in tarefas
    if isinstance(t, dict)
    and str(t.get("status", "")).strip().lower() == "queued"
]

objetos = drive_check.listdir("db/snapshots")
caminhos = {str(o.get("path", "")) for o in objetos if isinstance(o, dict)}
manager_check = SnapshotManager(
    drive_check, Path(os.environ.get("DATA_DIR", "/kaggle/working/xh-data")) / "tmp_15c_list"
)
manifests = manager_check.manifests()
snapshot_ok = False
if manifests:
    latest = manifests[0]
    file_path = latest.get("file")
    manifest_path = str(file_path).replace(".sqlite.gz", ".json") if file_path else ""
    snapshot_ok = bool(
        file_path and file_path in caminhos and manifest_path in caminhos
        and latest.get("epoch") is not None
    )

XH["etapa15c_lock_free"] = lock_free
XH["etapa15c_no_active_executors"] = not ativos
XH["etapa15c_no_running_tasks"] = not tarefas_em_execucao
XH["etapa15c_snapshot_ok"] = snapshot_ok
XH["etapa15c_ok"] = bool(
    lock_free and not ativos and not tarefas_em_execucao and snapshot_ok
    and XH.get("snapshot_bootstrap_ok") is True
)

print("===== CONFERÊNCIA FINAL REMOTA =====")
print("Locks brain:", len(brain_locks), "| livre:", lock_free)
print("Executores não terminais:", len(ativos))
print("Tarefas em execução/reivindicadas:", len(tarefas_em_execucao))
print("Tarefas queued (informativas):", len(queued))
print("Manifestos B2 válidos listados:", len(manifests))
print("Snapshot B2 encontrado:", snapshot_ok)

rec(
    "15C. Conferência final remota",
    "PASSOU" if XH["etapa15c_ok"] else "FALHOU",
    f"lock_livre={lock_free}; executores_ativos={len(ativos)}; "
    f"tarefas_em_execucao={len(tarefas_em_execucao)}; snapshot={snapshot_ok}"
)
if not XH["etapa15c_ok"]:
    raise RuntimeError(
        "Etapa 15C não aprovada. Não prossiga para a Etapa 15D nem inicialize o Worker."
    )
print("[PASSOU] Etapa 15C concluída em modo somente leitura.")


## CÉLULA DE CÓDIGO 18 — ETAPA 15D: Gate final e testes offline (somente leitura)

In [ ]:
# ============================================================
# ETAPA 15D — GATE FINAL
# Somente leitura; não inicia processos nem publica conteúdo.
# ============================================================

precisa_celula1()
import builtins

def avaliar_gate_15d(ctx, main_started=False):
    falhas = []
    flags = (
        "pronto", "turso_ready", "restore_ok", "lock_ok",
        "idempotency_dryrun_ok", "integridade_idempotencia_ok",
        "b2_read_ok", "sources_check_ok",
        "etapa15_preflight_ok", "snapshot_bootstrap_ok",
        "etapa15c_ok", "etapa15c_lock_free",
        "etapa15c_no_active_executors",
        "etapa15c_no_running_tasks", "etapa15c_snapshot_ok",
    )
    for nome in flags:
        if ctx.get(nome) is not True:
            falhas.append(f"flag não aprovada: {nome}")

    resultados = ctx.get("res", {})
    obrigatorios = (
        "1. Repositório", "1. Dependências",
        "2. Secrets obrigatórios", "2. Driver libsql",
        "2. Configuração Turso", "2. Configuração B2",
        "3. Healthcheck", "4. Imports/dependências",
        "5. Turso (leitura)", "6. Coordenador /api/state",
        "7. B2 autenticação/quota", "8. Lock do Brain",
        "9. Restore em cópia", "10. Fontes/scheduler",
        "10. Consulta de fontes", "11. Idempotência", "11. Integridade/idempotência",
        "15. Pré-validação final", "15B. Bootstrap B2", "15C. Conferência final remota",
    )
    for nome in obrigatorios:
        resultado = resultados.get(nome)
        if not resultado or resultado[0] != "PASSOU":
            falhas.append(f"resultado não aprovado: {nome}")

    if main_started:
        falhas.append("main() já foi iniciado neste kernel")
    return falhas

# Testes unitários puros: não usam Kaggle Secrets, rede, Turso, B2 ou arquivos.
_fixture_ok = {
    "pronto": True, "turso_ready": True, "restore_ok": True,
    "lock_ok": True, "idempotency_dryrun_ok": True, "integridade_idempotencia_ok": True,
    "b2_read_ok": True, "sources_check_ok": True,
    "etapa15_preflight_ok": True, "snapshot_bootstrap_ok": True,
    "etapa15c_ok": True, "etapa15c_lock_free": True,
    "etapa15c_no_active_executors": True,
    "etapa15c_no_running_tasks": True, "etapa15c_snapshot_ok": True,
    "res": {
        nome: ("PASSOU", "teste") for nome in (
            "1. Repositório", "1. Dependências",
            "2. Secrets obrigatórios", "2. Driver libsql",
            "2. Configuração Turso", "2. Configuração B2",
            "3. Healthcheck", "4. Imports/dependências",
            "5. Turso (leitura)", "6. Coordenador /api/state",
            "7. B2 autenticação/quota", "8. Lock do Brain",
            "9. Restore em cópia", "10. Fontes/scheduler", "10. Consulta de fontes",
            "11. Idempotência", "11. Integridade/idempotência",
            "15. Pré-validação final",
            "15B. Bootstrap B2", "15C. Conferência final remota",
        )
    },
}
assert avaliar_gate_15d(_fixture_ok) == [], "Teste positivo do gate falhou."

for _flag in ("turso_ready", "snapshot_bootstrap_ok", "etapa15c_ok", "sources_check_ok", "integridade_idempotencia_ok"):
    _fixture_bad = dict(_fixture_ok)
    _fixture_bad[_flag] = False
    assert avaliar_gate_15d(_fixture_bad), f"Gate não bloqueou {_flag}."

_fixture_bad_result = dict(_fixture_ok)
_fixture_bad_result["res"] = dict(_fixture_ok["res"])
_fixture_bad_result["res"]["15B. Bootstrap B2"] = ("FALHOU", "teste")
assert avaliar_gate_15d(_fixture_bad_result), "Gate não bloqueou bootstrap reprovado."
assert avaliar_gate_15d(_fixture_ok, main_started=True), "Gate não bloqueou main já iniciado."

print("[PASSOU] Teste unitário: cenário aprovado.")
print("[PASSOU] Testes unitários: flags críticas bloqueiam.")
print("[PASSOU] Teste unitário: bootstrap reprovado bloqueia.")
print("[PASSOU] Teste unitário: main duplicado bloqueia.")

# Avaliação real do estado das etapas anteriores.
_falhas_15d = avaliar_gate_15d(
    XH,
    main_started=getattr(builtins, "__XH_MAIN_STARTED__", False),
)
XH["etapa15d_ok"] = not _falhas_15d
rec(
    "15D. Gate final",
    "PASSOU" if XH["etapa15d_ok"] else "FALHOU",
    "todos os pré-requisitos e testes aprovados"
    if XH["etapa15d_ok"] else "; ".join(_falhas_15d)[:1000],
)

if _falhas_15d:
    print("\n[ BLOQUEADO ] Não altere INICIAR_KAGGLE_WORKER para True.")
    for _falha in _falhas_15d:
        print(" -", _falha)
    raise SystemExit("Etapa 15D reprovada; Worker permanece bloqueado.")

print("\n[PASSOU] ETAPA 15D — GATE FINAL APROVADO.")
print("[BLOQUEADO] Nenhum Worker/Brain/Executor foi iniciado por esta etapa.")
print("A célula de código 19 continua exigindo autorização manual explícita.")


## CÉLULA DE CÓDIGO 19 — ETAPA 15: Inicialização controlada (mantenha `INICIAR_KAGGLE_WORKER = False`)


In [ ]:
# ETAPA 15 — INICIALIZAÇÃO CONTROLADA
# Mantenha False durante diagnóstico/auditoria.
import os
import builtins

INICIAR_KAGGLE_WORKER = False

import json
import urllib.request

coord_url = os.environ.get("XH_COORD_URL", "").rstrip("/")
coord_token = os.environ.get("XH_COORD_TOKEN", "")
if not coord_url or not coord_token:
    raise RuntimeError("Coordenador não configurado; inicialização bloqueada.")

req = urllib.request.Request(
    coord_url + "/api/state",
    headers={
        "Authorization": "Bearer " + coord_token,
        "X-Requested-With": "XHK",
    },
)
try:
    with urllib.request.urlopen(req, timeout=20) as response:
        remote_state = json.loads(response.read().decode("utf-8"))
except Exception as exc:
    raise RuntimeError("Não foi possível validar a autorização no D1.") from exc

AUTORIZACAO_REMOTA = remote_state.get("launch_enabled") is True

if not INICIAR_KAGGLE_WORKER or not AUTORIZACAO_REMOTA:
    print("[BLOQUEADO] D1 não autorizou a inicialização.")
    print("Etapas 15B, 15C e 15D podem ser concluídas sem iniciar processos.")
    print("Nenhum Executor, Brain ou Engine foi iniciado.")
else:
    if XH.get("etapa15_preflight_ok") is not True:
        raise RuntimeError("Pré-validação da Etapa 15 não aprovada.")
    if XH.get("etapa15c_ok") is not True:
        raise RuntimeError("Etapa 15C não aprovada.")
    if XH.get("snapshot_bootstrap_ok") is not True:
        raise RuntimeError("Snapshot B2 não foi validado; inicialização bloqueada.")
    if XH.get("sources_check_ok") is not True:
        raise RuntimeError("Consulta da tabela sources não foi aprovada; inicialização bloqueada.")
    if XH.get("etapa15d_ok") is not True:
        raise RuntimeError("Etapa 15D não aprovou o gate final.")
    if XH.get("pronto") is not True:
        raise RuntimeError("Etapa 12 não liberou a operação.")
    if getattr(builtins, "__XH_MAIN_STARTED__", False):
        raise RuntimeError("main() já foi chamado neste kernel; não repetir.")
    if os.environ.get("XH_STORAGE_BACKEND", "").lower() != "b2":
        raise RuntimeError("Armazenamento B2 não está selecionado.")
    if os.environ.get("XH_STATE_BACKEND", "").lower() != "turso":
        raise RuntimeError("Turso não está selecionado como estado canônico.")

    from xhakinha.cloud.runner import main

    # Trava local antes de iniciar: evita chamadas duplicadas no mesmo kernel.
    builtins.__XH_MAIN_STARTED__ = True
    main()
